In [1]:
import pandas as pd
from datasets import load_dataset, Dataset

/opt/conda/envs/afrimmd/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test_data = pd.read_csv("finetune/test_predictions.csv", index_col=0)
allowed_languages = ["kik_Latn", "ibo_Latn", "yor_Latn", "hau_Latn", "amh_Ethi", "afr_Latn", "kin_Latn"]
test_data = test_data[test_data["language"].isin(allowed_languages)]
test_data.head()

,predictions,references,language,candidates
id,,,,
1,"[256025, 71112, 1022, 8423, 248272, 62, 14405,...",Mũndũ ũrarĩithia gĩcagi nĩ ahurũkaga thĩ kũger...,kik_Latn,Mũndũ ũrarũ na skithagi kĩa aurũkaga rũ yagere...
3,"[256025, 61493, 80, 35, 248116, 75223, 188, 35...",Imbwa y'umukara yambaye ishati y'ibara ry'umuh...,kin_Latn,Imbwa y'umukara yambaye ishati y'umuh ry'umuho...
5,"[256047, 48903, 201037, 62, 248105, 1825, 5, 8...",Nwa agbọghọ na-aga ịnwa ịkụ bọọlụ tennis .,ibo_Latn,Otu agbọghọ na-eto nwụ ịkụ bọọlụ tennis .
6,"[256051, 3963, 139108, 399, 3911, 3911, 133946...",Ọkùnrin kan tó ń ṣeré yìnyín sọ̀kalẹ̀ lórí òkè .,yor_Latn,Ọkùnrin kan ń ń rìneré lórínyín lórí̀kalẹ̀ lór...
8,"[256073, 3963, 139108, 11837, 11837, 94249, 39...",Ọkùnrin kan àti obìnrin kan ń wo ère aláwọ̀ pu...,yor_Latn,Ọkùnrin àti àti obìnrin kan ń wo àwòránre pupa...


In [3]:
raw_data = load_dataset("AfriMM/AFRICaption")["train"]

In [4]:
raw_data = raw_data.remove_columns(["id", "image_id"])
# Get all language columns except 'id', 'image_id', and 'eng'
lang_columns = [col for col in raw_data.column_names if col !='eng']

# Flatten the dataset: for each row, create a new row for each language
flattened_rows = []
for row in raw_data:
    for lang in lang_columns:
        flattened_rows.append({
            'eng': row['eng'],
            'local_caption': row[lang]
        })

combined_df = pd.DataFrame(flattened_rows)

In [5]:
merged_data = pd.merge(
    test_data, 
    combined_df,
    left_on='references',
    right_on='local_caption',
    how='left'
)

In [6]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3130 entries, 0 to 3129
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   predictions    3130 non-null   object
 1   references     3130 non-null   object
 2   language       3130 non-null   object
 3   candidates     3130 non-null   object
 4   eng            3130 non-null   object
 5   local_caption  3130 non-null   object
dtypes: object(6)
memory usage: 146.8+ KB


In [7]:
merged_data.drop_duplicates(subset=['references'], inplace=True)
merged_data.drop(columns=['predictions','local_caption', "language"], inplace=True)

In [8]:
merged_data

,references,candidates,eng
0,Mũndũ ũrarĩithia gĩcagi nĩ ahurũkaga thĩ kũger...,Mũndũ ũrarũ na skithagi kĩa aurũkaga rũ yagere...,A skier slides along a metal rail .
1,Imbwa y'umukara yambaye ishati y'ibara ry'umuh...,Imbwa y'umukara yambaye ishati y'umuh ry'umuho...,A brown dog in a pink shirt runs through a fie...
2,Nwa agbọghọ na-aga ịnwa ịkụ bọọlụ tennis .,Otu agbọghọ na-eto nwụ ịkụ bọọlụ tennis .,A young girl is about to attempt to hit a tenn...
3,Ọkùnrin kan tó ń ṣeré yìnyín sọ̀kalẹ̀ lórí òkè .,Ọkùnrin kan ń ń rìneré lórínyín lórí̀kalẹ̀ lór...,A man skiing down a snow covered mountain .
4,Ọkùnrin kan àti obìnrin kan ń wo ère aláwọ̀ pu...,Ọkùnrin àti àti obìnrin kan ń wo àwòránre pupa...,A man and a woman looking at a red sculpture .
...,...,...,...
3125,Umugore wari wambaye agapfukamunwa akubita umw...,Umugore w wambaye agapfukamunwa ybita umwana maso,A woman wearing a head covering holding an infant
3126,Twee honde aan leibande wat na mekaar toe span...,Twee honde wat traibande wat mekaar mekaar dra...,Two dogs on leashes straining toward each othe...
3127,Wani mutum yana tafiya a kan ƙasa mai tsawo ku...,Wani mutum yana tafiya da kan tafi mai tsawo k...,A man skateboards down a steep railing next to...
3128,kamwana gaarĩ na nguo cia kwĩhumba iria rĩarĩ ...,kamwana kaniniarĩ na nguo cia gũthhumba iria r...,a young boy wearing a colorful bathing suit sp...


In [9]:
merged_data = merged_data.rename(columns={"references": "ref", "eng":"src", "candidates": "mt"})

In [10]:
merged_data_dicts = merged_data.to_dict(orient="records")

In [11]:
from comet import download_model, load_from_checkpoint

model_path = download_model("masakhane/africomet-mtl")
model = load_from_checkpoint(model_path)
model_output = model.predict(merged_data_dicts, batch_size=16, gpus=1)
print(model_output)

Fetching 4 files: 100%|██████████| 4/4 [00:07<00:00,  2.00s/it]
Lightning automatically upgraded your loaded checkpoint from v1.9.5 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--masakhane--africomet-mtl/snapshots/c00016e2a120e8583b36c602d7be69ba1b45f088/checkpoints/model.ckpt`
Encoder model frozen.
You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_pr

Prediction([('scores', [0.26181215047836304, 0.6051399111747742, 0.5661895275115967, 0.45209214091300964, 0.5516659617424011, 0.19104453921318054, 0.6716539263725281, 0.3902663588523865, -0.05242149159312248, -0.06297816336154938, 0.41617104411125183, 0.09705983847379684, 0.4976620078086853, 0.5441562533378601, 0.3860210180282593, 0.14193668961524963, -0.15859876573085785, 0.45026323199272156, -0.07248911261558533, 0.5482356548309326, 0.7474896311759949, 0.7741848826408386, 0.8663567304611206, 0.11609955132007599, 0.7687346935272217, 0.8535468578338623, 0.22224001586437225, 0.8419792652130127, 0.18991288542747498, 0.16725942492485046, 0.12715092301368713, 0.5747005343437195, 0.41789907217025757, 0.5091909170150757, 0.46613258123397827, 0.7121192812919617, -0.09664267301559448, 0.15151390433311462, 0.43159884214401245, 0.5150694847106934, 0.44594842195510864, 0.5440855026245117, 0.38210153579711914, 0.514138400554657, 0.8166230916976929, 0.4045946002006531, -0.049023717641830444, 0.4112